In [0]:
%sql
USE default;

# Customer Support, Chatbot Effectiveness, and Cost Analysis

This notebook contains SQL queries used to analyze synthetic customer support data for XYZ Finance.

## Customer Support Performance

This section evaluates customer contact volume, support resolution rates, issue category trends, and customer satisfaction performance.

In [0]:
%sql
-- Q1: Which support channels receive the highest number of customer contacts?

SELECT
    channel,
    COUNT(*) AS total_contacts
FROM customer_contacts
GROUP BY channel
ORDER BY total_contacts DESC;

channel,total_contacts
Chatbot,30
Phone,26
Email,25
In-App Chat,19


In [0]:
%sql
-- Q2:What _is the overall support resolution rate across _all customer contacts?

SELECT 
     ROUND(
      SUM(CASE WHEN resolved_flag = TRUE THEN 1 
              ELSE 0 END)
               * 100.0 / COUNT(*),2) AS resolution_rate
               FROM customer_contacts;

resolution_rate
89.00


In [0]:
%sql
-- Q3:How does resolution rate differ _between chatbot, email, phone, _and in-app chat?

        SELECT
    channel,
    ROUND(
        SUM(
            CASE
                WHEN resolved_flag = TRUE THEN 1
                ELSE 0
            END
        ) * 100.00 / COUNT(*),
        2
    ) AS resolution_rate
FROM customer_contacts
GROUP BY channel
ORDER BY resolution_rate DESC;

channel,resolution_rate
Phone,96.15
Email,92.00
In-App Chat,89.47
Chatbot,80.00


In [0]:
%sql
-- Q4: Which issue categories have the highest contact _volume?
  
SELECT
    ic.issue_category_name,
    COUNT(*) AS contact_volume
FROM customer_contacts cc
JOIN issue_category ic
    ON cc.issue_category_id = ic.issue_category_id
GROUP BY ic.issue_category_name
ORDER BY contact_volume DESC;

issue_category_name,contact_volume
Direct Deposit,19
Card Issues,18
E-Transfer,17
Cashback Rewards,16
App Login,12
Subscription Billing,12
Fraud Security,6


In [0]:
%sql
-- Q5: Which issue categories have the lowest average CSAT score?

SELECT
    ic.issue_category_name,
    ROUND(AVG(cc.csat_score),2) AS avg_csat
FROM customer_contacts cc
JOIN issue_category ic
    ON cc.issue_category_id = ic.issue_category_id
GROUP BY ic.issue_category_name
ORDER BY avg_csat ASC;

issue_category_name,avg_csat
Fraud Security,61.49
E-Transfer,70.38
Subscription Billing,74.96
Cashback Rewards,75.22
Direct Deposit,76.65
Card Issues,78.67
App Login,81.32


## Chatbot Effectiveness

This section analyzes chatbot resolution performance, escalation rates, fallback behavior, and intent-level effectiveness.

In [0]:
%sql
-- Q1: What is the overall chatbot resolution/deflection rate?

SELECT
  ROUND(
    SUM(CASE WHEN bot_resolved_flag = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
    2
  ) AS chatbot_resolution_rate
FROM chatbot_interactions;

chatbot_resolution_rate
68.57


In [0]:
%sql
-- Q2: Which chatbot intents have the highest and lowest resolution rates?
SELECT
    ci.intent_name,
    ROUND(
        SUM(CASE WHEN ch.bot_resolved_flag = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS chatbot_resolution_rate
FROM chatbot_interactions ch
JOIN chatbot_intent ci
    ON ch.intent_id = ci.intent_id
GROUP BY ci.intent_name
ORDER BY chatbot_resolution_rate DESC;

intent_name,chatbot_resolution_rate
Password Reset,88.24
Rewards Inquiry,75.00
E-Transfer Issue,75.00
Direct Deposit Issue,66.67
Subscription Inquiry,66.67
Card Activation,58.33
Card Status,50.00
Fraud Investigation,0.00


In [0]:
%sql
-- Q3:What percentage of chatbot interactions result in agent handoff?

SELECT ROUND(
  SUM(CASE WHEN handoff_to_agent_flag = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
  2)
 AS agent_handoff_percentage
FROM chatbot_interactions;

agent_handoff_percentage
31.43


In [0]:
%sql
-- Q4: Which chatbot intents have the highest fallback rate?

SELECT ci.intent_name, ROUND(
  SUM(CASE WHEN fallback_flag = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*),2)
   AS fallback_rate FROM chatbot_interactions ch 
   JOIN chatbot_intent ci
    ON 
   ch.intent_id = ci.intent_id 
   GROUP BY ci.intent_name 
   ORDER BY fallback_rate DESC;

intent_name,fallback_rate
Card Status,50.00
Direct Deposit Issue,33.33
Card Activation,25.00
Rewards Inquiry,25.00
Fraud Investigation,25.00
E-Transfer Issue,25.00
Password Reset,5.88
Subscription Inquiry,0.00


In [0]:
%sql
-- Q5:How does chatbot resolution compare with agent-assisted resolution?

 SELECT
    ROUND(
        SUM(CASE WHEN ch.bot_resolved_flag = TRUE THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*),
        2
    ) AS chatbot_resolution_rate,

    ROUND(
        SUM(CASE 
                WHEN ch.handoff_to_agent_flag = TRUE 
                     AND cc.resolved_flag = TRUE 
                THEN 1 
                ELSE 0 
            END)
        * 100.0 
        / SUM(CASE WHEN ch.handoff_to_agent_flag = TRUE THEN 1 ELSE 0 END),
        2
    ) AS agent_assisted_resolution_rate

FROM chatbot_interactions ch
JOIN customer_contacts cc
    ON ch.contact_id = cc.contact_id;


chatbot_resolution_rate,agent_assisted_resolution_rate
68.57,86.36


## Cost and Operational Impact

This section evaluates support costs by channel, issue category, and interaction type.

In [0]:
%sql
-- Q1: What is the average cost per contact by support channel?

SELECT channel, 
ROUND(AVG(cost_per_contact),2) AS avg_cost_per_contact
FROM customer_contacts
GROUP BY channel
ORDER BY avg_cost_per_contact;

channel,avg_cost_per_contact
Chatbot,4.27
Email,7.62
Phone,8.08
In-App Chat,8.53


In [0]:
%sql
-- Q2: Which issue categories generate the highest total support cost?

SELECT ic.issue_category_name, 
ROUND(SUM(cost_per_contact),2) AS total_cost
FROM customer_contacts cc
JOIN issue_category ic
    ON cc.issue_category_id = ic.issue_category_id
GROUP BY ic.issue_category_name
ORDER BY total_cost DESC;

issue_category_name,total_cost
Direct Deposit,116.33
Card Issues,112.97
E-Transfer,109.32
Cashback Rewards,98.15
Fraud Security,88.87
Subscription Billing,85.76
App Login,79.29


In [0]:
%sql
-- Q3: What is the additional cost of handoffs relative to successful automation?

SELECT
    CASE
        WHEN ch.handoff_to_agent_flag = TRUE
        THEN 'Agent-assisted'
        ELSE 'Bot-only'
    END AS contact_type,

    COUNT(*) AS total_interactions,

    ROUND(AVG(cc.cost_per_contact),2) AS avg_cost_per_contact,

    ROUND(SUM(cc.cost_per_contact),2) AS total_cost

FROM chatbot_interactions ch
JOIN customer_contacts cc
    ON ch.contact_id = cc.contact_id

GROUP BY contact_type;


contact_type,total_interactions,avg_cost_per_contact,total_cost
Bot-only,48,6.98,334.80
Agent-assisted,22,6.10,134.15
